In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm import tqdm

In [ ]:
tqdm.pandas()

In [ ]:
df = pd.read_csv("../data/raw/spam_ham_dataset.csv")

In [ ]:
df.head()

In [ ]:
df = df.drop(columns=['Unnamed: 0', 'label'])

In [ ]:
df = df.rename(columns={'label_num': 'label'})

In [ ]:
df.info()

In [ ]:
#encoding string label to number (Only if label is string)
# from sklearn.preprocessing import LabelEncoder
# encoder = LabelEncoder()
# df['label'] = encoder.fit_transform(df['label'])

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df2 = pd.read_csv("../data/raw/quiz_questions.csv")

In [ ]:
df2.head()

In [ ]:
df2 = df2[df2['correct_answer'] == True].reset_index(drop=True)

In [ ]:
df2 = df2.drop(columns=['type','difficulty','category'])

In [ ]:
df2 = df2.rename(columns={'correct_answer': 'label'})

In [ ]:
df2 = df2.rename(columns={'question': 'text'})

In [ ]:
df2['label'] = 1

In [ ]:
df2.info()

In [ ]:
df3 = pd.read_csv("../data/raw/indian_email_dataset_100k.csv")

In [ ]:
df3.head()

In [ ]:
df3['label'] = df3['label'].map({'spam': 1, 'ham': 0})

In [ ]:
df3 = df3.rename(columns={'msg': 'text'})

In [ ]:
df3.info()

In [ ]:
df4 = pd.read_csv("../data/raw/email_spam.csv")

In [ ]:
df4.head()

In [ ]:
df4.info()

In [ ]:
df4['text'] = df4['title'] + " " + df4['text']

In [ ]:
df4=df4.drop(columns=['title'])

In [ ]:
df4 = df4.rename(columns={'type': 'label'})

In [ ]:
df4['label'] = df4['label'].map({'spam': 1, 'not spam': 0})

In [ ]:
df5 = pd.read_csv("../data/raw/filtered_business_leads.csv")

In [ ]:
df5.head()

In [ ]:
df5=df5.drop(columns=['type','queue'])

In [ ]:
df5 = df5.rename(columns={'query': 'text'})

In [ ]:
df5['label']=1

In [ ]:
df_final = pd.concat([df, df2, df3, df4, df5], ignore_index=True)

In [ ]:
# Combines them and adds a source tracker column
df_final = pd.concat([df, df2, df3, df4, df5], keys=['df1', 'df2', 'df3', 'df4', 'df5']) \
             .reset_index(level=0) \
             .rename(columns={'level_0': 'source_df'})


In [ ]:
df_final['label'].value_counts()

In [ ]:
df_final['text']=df_final['text'].astype(str)

In [ ]:
df_final = df_final.reset_index(drop=True)

In [ ]:
df_final.info()

In [ ]:
df_final.shape

In [ ]:
print(df_final['label'].value_counts())

Data Preprocessing

In [ ]:
import nltk
from nltk.corpus import stopwords
import string
nltk.download('punkt')

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
stop_words_set = set(stopwords.words('english'))
punctuation_set = set(string.punctuation)

In [ ]:
def transform_text(text):
    text = text.lower()
    if text.startswith("subject"):
        text = text.replace("subject", "", 1).strip()
    text = nltk.word_tokenize(text)

    y = [
        ps.stem(word) 
        for word in text 
        if word.isalnum() and word not in stop_words_set and word not in punctuation_set
    ]
            
    return " ".join(y)

In [ ]:
df_final.head()

In [ ]:
df_final['transformed_text'] = df_final['text'].progress_apply(transform_text)

In [ ]:
# # Injecting synthetic gibberish data into the dataframe so the model learns it
# gibberish_data = pd.DataFrame({
#     'text': ['asdfghjkl', 'qwertyuiop', 'zxcvbnm', 'mnbvcxz', 'lkjhgfdsa', 'poiuytrewq', 'qazwsxedc', 'plmoknijb','aushduksdfsdbvsdvbsdhbvysgvyh','sgdyugsacvasyc6agsccsn','326472639761579634975634957924657493'],
#     'label': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
# })

# # Appending the gibberish to the main dataset
# df = pd.concat([df, gibberish_data], ignore_index=True)

In [ ]:
df_final.info()

In [ ]:
df_final.to_csv('../data/cleaned/filtered_spam_data.csv', index=False)